## Notebook for experimenting with the prompt on the base model of Qwen3:1.7B

First let's load some data from the a Ping SQLite file using pandas.

In [ ]:
import sqlite3
import pandas as pd

# Configuration
DB_FILE = 'network_scans.db'

def load_data_to_pandas():
    try:
        # 1. Connect to the database
        conn = sqlite3.connect(DB_FILE)

        # 2. Define the SQL Query (Using LEFT JOINs to keep all data)
        #    Aliasing overlapping column names to ensure DataFrame columns are unique.
        query = """
        SELECT 
            -- Host Info
            h.hostId AS hostId,
            h.ipAddress AS ipAddress,
            h.hostnames AS hostnames,
            h.inferOS AS inferOS,
            h.macAddress AS macAddress,
            h.macVendor AS macVendor,
            
            -- Service Info
            s.serviceId AS serviceId,
            s.port AS port,
            s.protocol AS protocol,
            s.state AS state,
            s.serviceName AS serviceName,
            s.serviceProduct AS serviceProduct,
            s.serviceVersion AS serviceVersion,
            
            -- Vulnerability Info
            v.serviceId AS vuln_serviceId,
            v.dbSource AS dbSource,
            v.cveId AS cveId,
            v.description AS vuln_description
            
        FROM hosts h
        LEFT JOIN services s ON h.hostId = s.hostId
        LEFT JOIN vulnerabilities v ON s.serviceId = v.serviceId
        """

        # 3. Load directly into a DataFrame
        df = pd.read_sql_query(query, conn)
        
        return df

    except Exception as e:
        print(f"Error loading data: {e}")
        return None
    finally:
        # Good practice to close the connection, though pandas usually handles it safe enough
        if 'conn' in locals():
            conn.close()

# Usage
df = load_data_to_pandas()

if df is not None:
    # Preview the data
    print(df.head())

   hostId      ipAddress                        hostnames  \
0       1  192.168.100.1                               []   
1       2  192.168.100.4    ["OnePlus-7-Pro.localdomain"]   
2       3  192.168.100.6  ["Andrea-s-S20-FE.localdomain"]   
3       4  192.168.100.7             ["iPad.localdomain"]   
4       4  192.168.100.7             ["iPad.localdomain"]   

                                             inferOS         macAddress  \
0  [{"name": "FreeBSD 11.2-RELEASE", "accuracy": ...  BC:24:11:D1:6B:5F   
1                                                 []  C2:D3:C3:C1:6C:B0   
2                                                 []  7E:2C:E7:33:2B:CD   
3                                                 []  1E:EA:2C:D6:BF:8E   
4                                                 []  1E:EA:2C:D6:BF:8E   

                       macVendor  serviceId     port protocol state  \
0  Proxmox Server Solutions GmbH        1.0     53.0      tcp  open   
1                        Unknown        

Prepare data for insertion into the Jinja2 template.

In [23]:
import pandas as pd

def create_hierarchical_data(df):
    """
    Transforms the flat Pandas DataFrame into a list of hierarchical device dictionaries
    ready for insertion into the Jinja2 template.
    """
    hierarchical_data = []

    # Get unique hosts to iterate over
    unique_host_ids = df['hostId'].unique()

    for host_id in unique_host_ids:
        # Filter dataframe for the current host
        device_df = df[df['hostId'] == host_id]
        
        # 1. Extract Host Details
        # Take the first row since host details are identical for all rows of this group
        host_row = device_df.iloc[0]

        # 2. Extract Services
        # Select relevant columns, drop duplicates to handle the 'fan-out' from multiple vulns,
        # and remove rows where there is no service (port is NaN)
        service_cols = [
            'serviceId', 'port', 'protocol', 'state', 
            'serviceName', 'serviceProduct', 'serviceVersion'
        ]
        services_df = device_df[service_cols].drop_duplicates().dropna(subset=['port'])
        
        # fillna('') prevents "NaN" from appearing in your final output
        services_list = services_df.fillna('').to_dict('records')

        # 3. Extract Vulnerabilities
        # Select relevant columns and remove rows where there is no vulnerability (cveId is NaN)
        vuln_cols = ['vuln_serviceId', 'dbSource', 'cveId', 'vuln_description']
        vulns_df = device_df[vuln_cols].dropna(subset=['cveId'])
        
        vulns_list = vulns_df.fillna('').to_dict('records')

        # 4. Construct the Device Object
        # The keys here match the variables {{ key }} in your Jinja2 template
        device_obj = {
            "mac_vendor": host_row['macVendor'] if pd.notna(host_row['macVendor']) else "Unknown Vendor",
            "hostname": host_row['hostnames'] if pd.notna(host_row['hostnames']) else "Unknown Hostname",
            "ip_address": host_row['ipAddress'],
            "os_guess": host_row['inferOS'] if pd.notna(host_row['inferOS']) else "Unknown OS",
            "services": services_list,
            "vulnerabilities": vulns_list
        }

        hierarchical_data.append(device_obj)

    return hierarchical_data

# Usage Example:
devices_data = create_hierarchical_data(df)
print(devices_data[0]) # Inspect the first device object

{'mac_vendor': 'Proxmox Server Solutions GmbH', 'hostname': '[]', 'ip_address': '192.168.100.1', 'os_guess': '[{"name": "FreeBSD 11.2-RELEASE", "accuracy": "92"}]', 'services': [{'serviceId': 1.0, 'port': 53.0, 'protocol': 'tcp', 'state': 'open', 'serviceName': 'tcpwrapped', 'serviceProduct': '', 'serviceVersion': ''}], 'vulnerabilities': []}


Define the llm to test.

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen3:1.7b-fp16", temperature=0.6, top_p=0.95, top_k=20, reasoning=True, num_ctx=32768)  # Replace with your model name

Context length testing.

In [43]:
from transformers import AutoTokenizer

with open("test_output_full.md", "r") as file:
    data = file.read()

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")
tokens = tokenizer(data)

print(f"Number of tokens in the prompt: {len(tokens['input_ids'])}")

Token indices sequence length is longer than the specified maximum sequence length for this model (178771 > 131072). Running this sequence through the model will result in indexing errors


Number of tokens in the prompt: 178771


Let's try to prompt the base instruction tuned model with the first device using our jinja2 template!

*Credit [Alex Gonzalez](https://medium.com/@alecgg27895/jinja2-prompting-a-guide-on-using-jinja2-templates-for-prompt-management-in-genai-applications-e36e5c1243cf)*

In [42]:
from jinja2 import Environment, FileSystemLoader
import os
from langchain_core.prompts import ChatPromptTemplate

TEMPLATE = "prompt_template.j2"

SYSTEM_PROMPT = """You are a helpful and friendly home technology assistant. \
    Your goal is to analyze device reports and provide a simple, non-technical summary and a step-by-step guide for an average person.
    
    **Critical Constraints & Rules:**

1.  **No Jargon:** NEVER use technical terms like "CVE," "vulnerability," "exploit," "malicious," "command injection," "brute-force," "firmware," or "credentials."
2.  **Friendly Terminology:** Instead of the terms above, use words like "security issue," "glitch," "software bug," "login details," "update," or "settings."
3.  **Reassurance:** Always assure the user they have done nothing wrong.
4.  **Safe Password Advice:** NEVER suggest a specific password (like "SecurePass2023") or specific PIN (like "12345678"). Instead, tell the user to choose a "long, unique password" or a "random PIN." DO NOT use `(eg, ADVICE)` anywhere in your response.
5.  **Clickable Links:** If a device IP address is provided, you MUST format it as a clickable Markdown link, like this: `[192.168.1.1](http://192.168.1.1)`. Do not use backticks (`) or code blocks for the link.
6.  **Focus on the Fix:** Do not explain *how* the glitch works (e.g., do not mention "code execution"). Just explain that an update or setting change fixes it.
"""

def template_renderer(device_obj):
    env = Environment(loader=FileSystemLoader(os.path.dirname(TEMPLATE)))
    template = env.get_template(TEMPLATE)

    vulnerabilities_to_run = [ vuln for vuln in device_obj['vulnerabilities'] if vuln['dbSource'] == 'MITRE CVE' ]

    rendered_prompt = template.render(
        mac_vendor=device_obj['mac_vendor'],
        hostname=device_obj['hostname'],
        ip_address=device_obj['ip_address'],
        os_guess=device_obj['os_guess'],
        services=device_obj['services'],
        vulnerabilities=vulnerabilities_to_run
    )
    return rendered_prompt

print(template_renderer(devices_data[5]))

def langchain_prompter(devices_data):
    prompts = []

    for device in devices_data:
        prompt_text = template_renderer(device)
        chat_prompt = ChatPromptTemplate.from_messages(
            ('system', SYSTEM_PROMPT),
            ('user', prompt_text),
        )

        prompts.append(chat_prompt)
    return prompts

You have received the following report for a device with security issues. Walk the user through step-by-step on how to fix them. Provide clear instructions in a friendly format.

Remember:
- Do not use words like "vulnerability," "malicious," "firmware," or "credentials."
- Ensure all IP addresses are formatted as clickable links if the device would likely have a web interface. For example, most routers have web interfaces so in that case a router would be written like so: <a href="http://192.168.1.1">[HOSTNAME]</a>.
- Do not suggest specific examples for passwords or PINs.

## Device Scan Report
The mac vendor has been identified by nmap as: LG Innotek
The hostname has been identified by nmap as: ["LGwebOSTV.localdomain"]
You should refer to the device in your report using the hostname.
IP Address: 192.168.100.10
Nmap has inferred the OS of the device to most likely be: [{"name": "Linux 4.15 - 5.19", "accuracy": "100"}, {"name": "OpenWrt 21.02 (Linux 5.4)", "accuracy": "100"}, {"name"